# Regression


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor, StackingRegressor
from sklearn.neural_network import MLPRegressor

from xgboost import XGBRegressor


In [2]:
# 1. Data

def resolve_repo_root() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (
            (candidate / "data" / "creditsense-ai1215" / "credit_train.csv").exists()
            and (candidate / "submissions").exists()
            and (candidate / "task_b" / "artifacts").exists()
        ):
            return candidate
    raise FileNotFoundError("Could not resolve the project root for Task_B.ipynb.")


REPO_ROOT = resolve_repo_root()
DATA_DIR = REPO_ROOT / "data" / "creditsense-ai1215"
ARTIFACT_DIR = REPO_ROOT / "task_b" / "artifacts"
REGRESSION_ONLY_OUTPUT_PATH = ARTIFACT_DIR / "gukas_reg_submission.csv"
TASK_A_SUBMISSION_PATH = REPO_ROOT / "submissions" / "submission.csv"
FINAL_SUBMISSION_PATH = REPO_ROOT / "submissions" / "submission_Dachi_class_Guka_reg.csv"
NOTEBOOK_FINAL_COPY_PATH = REPO_ROOT / "task_b" / "notebooks" / "submission_DCGR2.csv"

df = pd.read_csv(DATA_DIR / "credit_train.csv")

X  = df.drop(['RiskTier', 'InterestRate'], axis=1)
y_reg = df['InterestRate']

X_train, X_test, y_train, y_test = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

numeric_features = X_train.select_dtypes(include=["number"]).columns.tolist()
categorical_features = X_train.select_dtypes(exclude=["number"]).columns.tolist()


# 2. Preprocessing
basic_numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

basic_categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

basic_preprocessor = ColumnTransformer([
    ("num", basic_numeric_preprocessor, numeric_features),
    ("cat", basic_categorical_preprocessor, categorical_features)
])

# For neural network:
nn_numeric_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

nn_categorical_preprocessor = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

nn_preprocessor = ColumnTransformer([
    ("num", nn_numeric_preprocessor, numeric_features),
    ("cat", nn_categorical_preprocessor, categorical_features)
])


In [4]:

# 3. 5 Base Models

xgb_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", XGBRegressor(
        n_estimators=600,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror",
        random_state=42
    ))
])

rf_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=600,
        max_depth=None,
        random_state=42,
        n_jobs=-1
    ))
])

hgb_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", HistGradientBoostingRegressor(
        max_iter=600,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    ))
])

lr_model = Pipeline([
    ("preprocess", basic_preprocessor),
    ("model", LinearRegression())
])

mlp_model = Pipeline([
    ("preprocess", nn_preprocessor),
    ("model", MLPRegressor(
        hidden_layer_sizes=(256, 128, 64),
        activation="relu",
        solver="adam",
        alpha=0.0001,
        batch_size="auto",
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        random_state=42
    ))
])


# Final Combined Model
base_models = [
    ("xgb", xgb_model),
    ("rf", rf_model),
    ("hgb", hgb_model),
    ("lr", lr_model),
    ("mlp", mlp_model),
]

stack_model = StackingRegressor(
    estimators=base_models,
    final_estimator=Ridge(alpha=1.0),
    cv=5,
    n_jobs=-1,
    passthrough=False
)


# Train+prediction
stack_model.fit(X_train, y_train)


y_pred = stack_model.predict(X_test)


# Results
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Stacked Model Results")
print(f"MSE  : {mse:.4f}")
print(f"RMSE : {rmse:.4f}")
print(f"MAE  : {mae:.4f}")
print(f"R2   : {r2:.4f}")

Stacked Model Results
MSE  : 2.8780
RMSE : 1.6965
MAE  : 1.3422
R2   : 0.8266


In [5]:
test_df = pd.read_csv(DATA_DIR / "credit_test.csv")

X_test_final = test_df
y_test_final_pred = stack_model.predict(X_test_final)

test_submission = pd.DataFrame({
    "InterestRate": y_test_final_pred
})
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
test_submission.to_csv(REGRESSION_ONLY_OUTPUT_PATH, index=False)


In [6]:
last_submission = pd.read_csv(TASK_A_SUBMISSION_PATH)
y_test_final_pred = np.clip(y_test_final_pred, 4.99, 35.99)
last_submission["InterestRate"] = y_test_final_pred

FINAL_SUBMISSION_PATH.parent.mkdir(parents=True, exist_ok=True)
NOTEBOOK_FINAL_COPY_PATH.parent.mkdir(parents=True, exist_ok=True)
last_submission.to_csv(FINAL_SUBMISSION_PATH, index=False)
last_submission.to_csv(NOTEBOOK_FINAL_COPY_PATH, index=False)
